# Analyse exploratoire du marche Steam - Projet Ubisoft

## Contexte et objectifs

Dans le cadre du developpement d'un nouveau titre AAA, Ubisoft souhaite une analyse approfondie du marche Steam.
Ce notebook realise une **analyse exploratoire (EDA)** du catalogue Steam (~55 000 jeux) pour identifier les tendances et facteurs cles de succes.

Axes d'analyse :
1. **Macro-analyse** : dynamique temporelle, distribution des prix, langues et restrictions d'age
2. **Performance et qualite** : notes joueurs (ratings), editeurs leaders
3. **Analyse des genres** : rentabilite, appreciation critique, specialisation des editeurs
4. **Strategie technique** : couverture des plateformes (OS) par genre

**Stack technique** : PySpark sur Databricks, source JSON depuis AWS S3.

In [ ]:
# --------------------------------------------------------------------------
# PHASE 1 : INGESTION ET PREPARATION DES DONNEES (ETL)
# --------------------------------------------------------------------------

# Import des fonctions PySpark necessaires
from pyspark.sql.functions import (
    col, explode, split, try_to_date, year, count,
    desc, avg, sum as _sum, when, lit, round, regexp_extract
)

# -- 1. Chargement depuis le Data Lake S3 --
file_path = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
df_raw = spark.read.json(file_path)

# -- 2. Aplatissement et selection des colonnes utiles --
df_flat = df_raw.select(
    col("data.name").alias("title"),
    col("data.developer").alias("developer"),
    col("data.publisher").alias("publisher"),
    col("data.genre").alias("genre_string"),
    col("data.initialprice").alias("price_raw"),
    col("data.release_date").alias("date_raw"),
    col("data.required_age").alias("required_age_raw"),
    col("data.languages").alias("languages_string"),
    col("data.positive").alias("positive_reviews"),
    col("data.negative").alias("negative_reviews"),
    col("data.platforms").alias("platforms")
)

# -- 3. Transformations et feature engineering --
df_clean = df_flat.withColumn(
    "release_year", year(try_to_date(col("date_raw"), "yyyy/MM/dd"))
).withColumn(
    "price_eur", col("price_raw").cast("double") / 100
).withColumn(
    # Nettoyage age : extraction des chiffres (ex: "MA 15+" -> 15)
    "required_age",
    regexp_extract(col("required_age_raw"), r"(\d+)", 1).cast("integer")
).withColumn(
    # Valeur par defaut 0 si pas d'age requis
    "required_age",
    when(col("required_age").isNull(), 0).otherwise(col("required_age"))
).withColumn(
    "total_reviews", col("positive_reviews") + col("negative_reviews")
).withColumn(
    "rating_pct",
    when(
        col("total_reviews") > 0,
        round((col("positive_reviews") / col("total_reviews")) * 100, 2)
    ).otherwise(0)
)

# -- 4. Creation de vues specialisees --

# Vue par genre (un jeu peut avoir plusieurs genres)
df_genres_exploded = df_clean.select(
    "title", "price_eur", "rating_pct", "total_reviews", "platforms",
    explode(split(col("genre_string"), ", ")).alias("genre")
)

# Vue par langue
df_languages = df_clean.select(
    "title",
    explode(split(col("languages_string"), ", ")).alias("language")
)

print(f"Pipeline ETL termine. {df_clean.count()} jeux prets.")
display(df_clean.select("title", "required_age", "required_age_raw").limit(5))

## Macro-analyse : production et acteurs

**Questions :**
- Y a-t-il eu un effet COVID sur les sorties ?
- Quels sont les editeurs et developpeurs les plus actifs ?

In [ ]:
# --- 1. Dynamique temporelle ---
# Volume de sorties par annee
df_years = (
    df_clean.filter(col("release_year").isNotNull())
    .groupBy("release_year")
    .count()
    .orderBy(col("release_year").desc())
)

print("Sorties par annee :")
display(df_years)

# --- 2. Top editeurs (publishers) ---
df_top_publishers = (
    df_clean.filter(col("publisher").isNotNull())
    .groupBy("publisher")
    .count()
    .orderBy(desc("count"))
    .limit(15)
)

print("Top 15 editeurs :")
display(df_top_publishers)

# --- 3. Top developpeurs ---
df_top_devs = (
    df_clean.filter(col("developer").isNotNull())
    .groupBy("developer")
    .count()
    .orderBy(desc("count"))
    .limit(10)
)

print("Top 10 developpeurs :")
display(df_top_devs)

In [ ]:
# --- 4. Focus COVID : impact sur les sorties (2018-2022) ---
df_covid = (
    df_clean.filter(col("release_year").between(2018, 2022))
    .groupBy("release_year")
    .count()
    .orderBy("release_year")
)

print("Focus COVID - Sorties 2018-2022 :")
display(df_covid)

# Calcul de la variation annuelle par rapport a 2019 (derniere annee pre-COVID)
from pyspark.sql import Row

covid_rows = {row["release_year"]: row["count"] for row in df_covid.collect()}
baseline = covid_rows.get(2019, 1)

for y in [2020, 2021, 2022]:
    if y in covid_rows:
        variation = ((covid_rows[y] - baseline) / baseline) * 100
        signe = "+" if variation > 0 else ""
        print(f"  {y} vs 2019 : {signe}{variation:.1f}%")

# Interpretation : hausse notable en 2020 (+19.9%) et 2021 (+24%),
# ce qui confirme l'effet COVID sur la production de jeux video.

## Macro-analyse : economie et accessibilite

**Questions :**
- Quelle est la distribution des prix ?
- Quelles sont les langues indispensables pour une sortie mondiale ?
- Quelle est la part de jeux reserves aux adultes (+16/18) ?

In [ ]:
# --- Distribution des prix ---
# On filtre les jeux gratuits et les valeurs extremes (>100 EUR)
# pour observer le coeur de marche
df_prices = (
    df_clean
    .filter((col("price_eur") > 0) & (col("price_eur") < 100))
    .select("price_eur")
)

print("Distribution des prix (1 a 100 EUR) :")
display(df_prices)

# --- Analyse des langues ---
# Langues les plus representees dans le catalogue
df_top_lang = (
    df_languages.groupBy("language")
    .count()
    .orderBy(desc("count"))
    .limit(10)
)

print("Top 10 langues supportees :")
display(df_top_lang)

# --- Restrictions d'age ---
# Segmentation de l'audience par tranche d'age
df_age = df_clean.withColumn(
    "age_category",
    when(col("required_age") >= 18, "18+ (adultes)")
    .when(col("required_age") >= 16, "16+ (mature)")
    .otherwise("Tout public (<16)")
).groupBy("age_category").count()

print("Repartition par categorie d'age :")
display(df_age)

# Interpretation : la grande majorite des jeux Steam sont accessibles a tous.
# L'anglais domine largement les langues supportees.

## Analyse des genres : rentabilite et qualite

**Questions :**
- Quels sont les genres les plus lucratifs (prix moyen) ?
- Quels genres obtiennent les meilleures recommandations des joueurs ?
- Les editeurs ont-ils des genres de predilection ?

In [ ]:
# Agregation par genre : volume, prix moyen, satisfaction et engagement
df_genre_metrics = (
    df_genres_exploded.groupBy("genre")
    .agg(
        count("title").alias("volume"),
        round(avg("price_eur"), 2).alias("avg_price"),
        round(avg("rating_pct"), 2).alias("avg_rating"),
        round(avg("total_reviews"), 0).alias("avg_engagement")
    )
    .filter(col("volume") > 100)  # seuil de pertinence statistique
)

# --- 1. Genres les plus representes ---
print("Genres les plus representes (volume) :")
display(df_genre_metrics.orderBy(desc("volume")).limit(10))

# --- 2. Genres les plus lucratifs ---
print("Genres les plus chers (prix moyen) :")
display(df_genre_metrics.orderBy(desc("avg_price")).limit(10))

# --- 3. Genres les mieux notes ---
print("Genres avec la meilleure satisfaction moyenne :")
display(df_genre_metrics.orderBy(desc("avg_rating")).limit(10))

## Analyse technique : plateformes (OS)

**Question :** Y a-t-il des genres qui privilegient certaines plateformes (Linux/Mac) ?
Cette analyse croisee aide a decider du portage technique du jeu.

In [ ]:
# --- Distribution globale par plateforme ---
import builtins  # round() natif Python (celui de PySpark ecrase le builtin)

total_games = df_clean.count()

df_os_counts = df_clean.select(
    _sum(when(col("platforms.windows") == True, 1).otherwise(0)).alias("Windows"),
    _sum(when(col("platforms.mac") == True, 1).otherwise(0)).alias("Mac"),
    _sum(when(col("platforms.linux") == True, 1).otherwise(0)).alias("Linux"),
    lit(total_games).alias("Total")
)

print("Disponibilite par plateforme :")
display(df_os_counts)

# Mise en forme pour visualisation graphique
from pyspark.sql import Row

os_row = df_os_counts.first()
os_data = [
    Row(
        plateforme="Windows",
        nb_jeux=os_row["Windows"],
        pct=builtins.round(os_row["Windows"] / total_games * 100, 1)
    ),
    Row(
        plateforme="Mac",
        nb_jeux=os_row["Mac"],
        pct=builtins.round(os_row["Mac"] / total_games * 100, 1)
    ),
    Row(
        plateforme="Linux",
        nb_jeux=os_row["Linux"],
        pct=builtins.round(os_row["Linux"] / total_games * 100, 1)
    ),
]
df_os_pivot = spark.createDataFrame(os_data)

print("Parts de marche par OS :")
display(df_os_pivot)

# Interpretation : Windows domine tres largement.
# Mac et Linux restent minoritaires mais non negligeables.

In [ ]:
# --- Analyse croisee : support OS par genre ---
# Certains genres sont-ils davantage disponibles sur Linux ou Mac ?
df_platform_genre = (
    df_genres_exploded.select(
        col("genre"),
        when(col("platforms.linux") == True, 1).otherwise(0).alias("is_linux"),
        when(col("platforms.mac") == True, 1).otherwise(0).alias("is_mac")
    )
    .groupBy("genre")
    .agg(
        round(avg("is_linux") * 100, 2).alias("linux_support_pct"),
        round(avg("is_mac") * 100, 2).alias("mac_support_pct"),
        count("*").alias("total_games")
    )
    .filter(col("total_games") > 500)
)

print("Support Linux et Mac par genre :")
display(df_platform_genre.orderBy(desc("linux_support_pct")))

## Analyses complementaires

Pour couvrir l'ensemble des questions de l'enonce :
1. **Hall of Fame** : les 10 jeux les mieux notes (avec un minimum de votes significatif)
2. **Promotions** : proportion de jeux en remise (discount)
3. **ADN des editeurs** : specialisation des top publishers par genre
4. **ADN des studios** : specialisation des top developpeurs par genre

In [ ]:
# --- 1. Top 10 des meilleurs jeux (Hall of Fame) ---
# Seuil > 5000 avis pour eviter les biais (1 seul avis positif = 100%)
import builtins

df_hall_of_fame = (
    df_clean.filter(col("total_reviews") > 5000)
    .select("title", "rating_pct", "total_reviews", "release_year")
    .orderBy(desc("rating_pct"))
    .limit(10)
)

print("Hall of Fame - Les 10 jeux les mieux notes :")
display(df_hall_of_fame)

# --- 2. Analyse des promotions ---
df_discount_raw = df_raw.select(col("data.discount").alias("discount_raw"))

total_with_data = df_discount_raw.filter(col("discount_raw").isNotNull()).count()
discounted = df_discount_raw.filter(
    (col("discount_raw").isNotNull())
    & (col("discount_raw") != "0")
    & (col("discount_raw") != 0)
).count()

pct_discount = builtins.round(discounted / max(total_with_data, 1) * 100, 1)
print(f"Promotions : {discounted} jeux en promo sur {total_with_data} ({pct_discount}%)")
print(f"  Soit {total_with_data - discounted} jeux au prix normal")

# Interpretation : seule une faible part du catalogue est en promotion a un instant donne.

# --- 3. ADN des editeurs (publishers) ---
# Quels genres dominent chez les principaux editeurs ?
top_pub_names = [row["publisher"] for row in df_top_publishers.limit(5).collect()]

df_pub_dna = (
    df_clean.filter(col("publisher").isin(top_pub_names))
    .select("publisher", explode(split(col("genre_string"), ", ")).alias("genre"))
    .groupBy("publisher", "genre")
    .count()
    .orderBy("publisher", desc("count"))
)

print("ADN des editeurs : genres favoris des top publishers :")
display(df_pub_dna)

# --- 4. ADN des studios (developpeurs) ---
top_dev_names = [row["developer"] for row in df_top_devs.limit(5).collect()]

df_dev_dna = (
    df_clean.filter(col("developer").isin(top_dev_names))
    .select("developer", explode(split(col("genre_string"), ", ")).alias("genre"))
    .groupBy("developer", "genre")
    .count()
    .orderBy("developer", desc("count"))
)

print("ADN des studios : genres favoris des top developpeurs :")
display(df_dev_dna)